# MTLnet — Colab Study Runner

Parallel worker for the claim-driven ablation in `docs/studies/`.

**Flow:**
1. Run **Bootstrap** once per session.
2. Pick a `test_id` the primary machine assigned to Colab (see `/study status` on the Mac).
3. Run it. The script packages the output under Drive.
4. Sync Drive locally and run `/study import` to archive into `docs/studies/results/`.

## ① Bootstrap

In [ ]:
# Mount Drive + clone repo so the runner is available
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/VitorHugoOli/PoiMtlNet.git /content/PoiMtlNet || git -C /content/PoiMtlNet pull --ff-only
!git -C /content/PoiMtlNet log --oneline -3

In [ ]:
# Install deps (idempotent; skip pip if a prior cell already did it)
!python /content/PoiMtlNet/scripts/study/colab_runner.py bootstrap \
    --drive-root /content/drive/MyDrive/mestrado/PoiMtlNet

## ② Pick a test

Tests must already be enrolled in `docs/studies/state.json` on the Mac and pushed to `main`. Use the cell below to see what's planned.

In [ ]:
!python /content/PoiMtlNet/scripts/study/colab_runner.py list

In [ ]:
# ── edit here ────────────────────────────────────────────────────────────
PHASE   = 'P1'
TEST_ID = 'P1_AL_smoke'   # replace with the id you were asked to run
# ─────────────────────────────────────────────────────────────────────────

# Dry-run first (prints the command, no training)
!python /content/PoiMtlNet/scripts/study/colab_runner.py run \
    --phase {PHASE} --test-id {TEST_ID} --dry-run

## ③ Run it

The tarball lands under `/content/drive/MyDrive/mestrado/PoiMtlNet/results/_study_artifacts/`.
Sync Drive locally (the Drive app does this), then on the Mac:

```bash
tar -xzf <tarball>.tar.gz -C /tmp
python scripts/study/study.py import \
    --run-dir /tmp/<run_dir_name> \
    --phase P1 --test-id <test_id> --claims C01 C11
```

In [ ]:
!python /content/PoiMtlNet/scripts/study/colab_runner.py run \
    --phase {PHASE} --test-id {TEST_ID}

## ④ (Optional) Run multiple tests in sequence

Just repeat section ③ with a new `TEST_ID`. Each run packages its own tarball.
Training times vary by config (Alabama MTL ~20 min / fold, Florida ~4 h / fold on T4).